In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier # For RFC
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [ ]:
df = pd.read_csv("house_prices_large_dataset.csv")

In [ ]:
features = df.columns
print (features)
print (type(features))
df.head(5)



In [ ]:
df.describe()
df.info()

In [ ]:
pd.crosstab(df["Property_Type"], df["Floor"].isna())

In [ ]:
numeric_cols = [
    "Area_sqft",
    "Bedrooms",
    "Bathrooms",
    "Garage",
    "Year_Built",
    "Floor",
    "Distance_to_CityCenter_km",
    "Price_USD"
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x="Area_sqft",
    y="Price_USD",
    alpha=0.3
)

plt.title("House Area vs Price")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="Distance_to_CityCenter_km",
    y="Price_USD",
    alpha=0.3
)

plt.title("Distance to City Center vs Price")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x="Property_Type",
    y="Price_USD"
)

plt.xticks(rotation=45)
plt.title("Price by Property Type")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="Location",
    y="Price_USD"
)

plt.xticks(rotation=90)
plt.title("Price by Location")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(df["Price_USD"], kde=True)

plt.title("Distribution of House Prices")
plt.show()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded = encoder.fit_transform(
    df[["Location", "Property_Type"]]
)

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Location", "Property_Type"]),
    index=df.index
)

print(encoded_df.head())

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Price_USD", "House_ID"])
y = df["Price_USD"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
numeric_features = [
    "Area_sqft",
    "Bedrooms",
    "Bathrooms",
    "Garage",
    "Year_Built",
    "Floor",
    "Distance_to_CityCenter_km"
]

categorical_features = [
    "Location",
    "Property_Type"
]


In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)


# Evaluate Linear Regression
linear_mae = mean_absolute_error(
    y_test,
    linear_predictions
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions
    )
)

linear_r2 = r2_score(
    y_test,
    linear_predictions
)

print("\n===== LINEAR REGRESSION =====")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R²:", linear_r2)

In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)


# Evaluate Random Forest
rf_mae = mean_absolute_error(
    y_test,
    rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_predictions
    )
)

rf_r2 = r2_score(
    y_test,
    rf_predictions
)

print("\n===== RANDOM FOREST =====")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R²:", rf_r2)

In [ ]:
gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)

gb_predictions = gb_model.predict(X_test)


# Evaluate Gradient Boosting
gb_mae = mean_absolute_error(
    y_test,
    gb_predictions
)

gb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        gb_predictions
    )
)

gb_r2 = r2_score(
    y_test,
    gb_predictions
)

print("\n===== GRADIENT BOOSTING =====")
print("MAE:", gb_mae)
print("RMSE:", gb_rmse)
print("R²:", gb_r2)

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],

    "MAE": [
        linear_mae,
        rf_mae,
        gb_mae
    ],

    "RMSE": [
        linear_rmse,
        rf_rmse,
        gb_rmse
    ],

    "R2": [
        linear_r2,
        rf_r2,
        gb_r2
    ]
})

print("\n===== MODEL COMPARISON =====")
print(results.sort_values("RMSE"))

In [ ]:
# Get trained preprocessing and model
rf_preprocessor = rf_model.named_steps["preprocessor"]
rf_estimator = rf_model.named_steps["model"]

# Get feature names after one-hot encoding
feature_names = rf_preprocessor.get_feature_names_out()

# Get Random Forest importance
importances = rf_estimator.feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

print("\n===== TOP FEATURES =====")
print(importance_df.head(20))


# Plot feature importance
plt.figure(figsize=(10, 7))

sns.barplot(
    data=importance_df.head(15),
    x="Importance",
    y="Feature"
)

plt.title("Top Features Driving House Prices")
plt.show()

In [ ]:

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=y_test,
    y=gb_predictions,
    alpha=0.4
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted House Prices")

# Perfect prediction line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)

plt.show()

In [27]:
prediction_df = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": gb_predictions
})

print("\n===== SAMPLE PREDICTIONS =====")
print(prediction_df.head(10))


===== SAMPLE PREDICTIONS =====
   Actual Price  Predicted Price
0       2455132     2.320089e+06
1       1077044     1.197509e+06
2       1196570     1.045988e+06
3       1116835     1.024429e+06
4        542284     4.837414e+05
5        227637     2.549859e+05
6       1755030     1.740521e+06
7       1943809     1.783172e+06
8       1144273     1.220189e+06
9       1745110     1.848040e+06
